# RLO Experiments - Optimized for NVIDIA RTX 5090 (Windows Compatible)

**Key Optimizations:**
- Large batch sizes (512-1024)
- bfloat16 mixed precision
- TF32 for tensor cores
- Optimized DataLoader
- GPU-side preprocessing

**Note:** torch.compile is disabled on Windows (Triton not supported)

In [ ]:
# =============================================================================
# IMPORTS AND GPU OPTIMIZATION
# =============================================================================

import os
import sys
import time
import math
import random
import json
import platform
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as T
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# =============================================================================
# GPU OPTIMIZATIONS
# =============================================================================

# TF32 for massive speedup on Ampere+ GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.set_float32_matmul_precision('high')

# Check if we can use torch.compile (NOT on Windows)
IS_WINDOWS = platform.system() == 'Windows'
USE_COMPILE = False  # Disable on Windows due to Triton issues

if IS_WINDOWS:
    print("⚠️  Windows detected: torch.compile disabled (Triton not supported)")
    print("   Other optimizations (TF32, bfloat16, large batch) still provide good speedup!\n")

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"Memory: {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
    print(f"PyTorch: {torch.__version__}")
    
    GPU_MEM_GB = props.total_memory / 1e9
    if GPU_MEM_GB >= 24:
        REC_BATCH = 512
    elif GPU_MEM_GB >= 16:
        REC_BATCH = 256
    else:
        REC_BATCH = 128
    print(f"Recommended batch size: {REC_BATCH}")

IMAGENET_PATH = Path(r"C:\Users\PC\.cache\huggingface\datasets\imagenet-1k")
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# =============================================================================
# ALL OPTIMIZERS
# =============================================================================

class RLO(Optimizer):
    """Riemannian Lyapunov Optimizer."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.1, 
                 belief_coef=0.1, eps=1e-8):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay, 
                       belief_coef=belief_coef, eps=eps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr, wd = group["lr"], group["weight_decay"]
            beta1, beta2 = group["betas"]
            belief, eps = group["belief_coef"], group["eps"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state["exp_avg"] = torch.zeros_like(p)

                m = state["exp_avg"]

                if wd != 0.0:
                    p.mul_(1.0 - lr * wd)

                c = beta1 * m + (1.0 - beta1) * g
                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                d = c.sign() + belief * (delta / delta_norm)

                p.add_(d, alpha=-lr)
                m.mul_(beta2).add_(g, alpha=(1.0 - beta2))

        return loss


class RLO_LambdaA(Optimizer):
    """RLO with adaptive preconditioning."""
    def __init__(self, params, lr=1e-4, beta1=0.9, beta2=0.99, beta3=0.999,
                 weight_decay=0.1, lambda_b=0.1, eps=1e-8, gamma=5.0):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, beta3=beta3,
                       weight_decay=weight_decay, lambda_b=lambda_b,
                       eps=eps, gamma=gamma)
        super().__init__(params, defaults)
        self._init_sqrt_dim()

    def _init_sqrt_dim(self):
        total = sum(p.numel() for g in self.param_groups for p in g["params"])
        self.sqrt_dim = math.sqrt(total)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        all_smooth_pre, all_belief, all_params = [], [], []

        for group in self.param_groups:
            eps, gamma = group["eps"], group["gamma"]
            beta1, beta2, beta3 = group["beta1"], group["beta2"], group["beta3"]
            lambda_b = group["lambda_b"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                
                if len(state) == 0:
                    state["m"] = torch.zeros_like(p)
                    state["s"] = torch.zeros_like(p)

                m, s = state["m"], state["s"]
                s.mul_(beta3).addcmul_(g, g, value=(1.0 - beta3))

                c = beta1 * m + (1.0 - beta1) * g
                smooth = torch.tanh(gamma * c)
                smooth_pre = smooth / (s.sqrt() + eps)

                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                belief = lambda_b * (delta / delta_norm)

                all_smooth_pre.append(smooth_pre)
                all_belief.append(belief)
                all_params.append((p, group))

        if not all_params:
            return loss

        s_norm_sq = sum((sp * sp).sum() for sp in all_smooth_pre)
        s_norm = s_norm_sq.sqrt().clamp(min=1e-8)
        scale = self.sqrt_dim / s_norm

        for (p, group), sp, b in zip(all_params, all_smooth_pre, all_belief):
            lr, wd, beta2 = group["lr"], group["weight_decay"], group["beta2"]
            d = scale * sp + b
            state = self.state[p]

            if wd != 0.0:
                p.mul_(1.0 - lr * wd)

            p.add_(d, alpha=-lr)
            state["m"].mul_(beta2).add_(p.grad, alpha=(1.0 - beta2))

        return loss


class SmoothLiftedRLO(Optimizer):
    """Second-order lifted RLO."""
    def __init__(self, params, lr=1e-4, beta1=0.9, beta2=0.99, eta=0.3,
                 weight_decay=0.1, lambda_b=0.1, eps=1e-8, gamma=5.0):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eta=eta,
                       weight_decay=weight_decay, lambda_b=lambda_b,
                       eps=eps, gamma=gamma)
        super().__init__(params, defaults)
        self._init_sqrt_dim()

    def _init_sqrt_dim(self):
        total = sum(p.numel() for g in self.param_groups for p in g["params"])
        self.sqrt_dim = math.sqrt(total)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        all_s, all_b, all_params = [], [], []

        for group in self.param_groups:
            eps, gamma = group["eps"], group["gamma"]
            beta1, lambda_b = group["beta1"], group["lambda_b"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                
                if len(state) == 0:
                    state["m"] = torch.zeros_like(p)
                    state["v"] = torch.zeros_like(p)

                m = state["m"]
                c = beta1 * m + (1.0 - beta1) * g
                s = torch.tanh(gamma * c)

                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                b = lambda_b * (delta / delta_norm)

                all_s.append(s)
                all_b.append(b)
                all_params.append((p, group))

        if not all_params:
            return loss

        s_norm_sq = sum((s * s).sum() for s in all_s)
        s_norm = s_norm_sq.sqrt().clamp(min=1e-8)
        scale = self.sqrt_dim / s_norm

        for (p, group), s, b in zip(all_params, all_s, all_b):
            lr, wd, eta, beta2 = group["lr"], group["weight_decay"], group["eta"], group["beta2"]
            d = scale * s + b
            state = self.state[p]
            m, v = state["m"], state["v"]

            if wd != 0.0:
                p.mul_(1.0 - lr * wd)

            v.mul_(1.0 - eta).add_(d, alpha=eta)
            p.add_(v, alpha=-lr)
            m.mul_(beta2).add_(p.grad, alpha=(1.0 - beta2))

        return loss


class Lion(Optimizer):
    """Lion optimizer."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr, wd = group['lr'], group['weight_decay']
            beta1, beta2 = group['betas']
            
            for p in group['params']:
                if p.grad is None:
                    continue

                if wd != 0.0:
                    p.mul_(1.0 - lr * wd)

                g = p.grad
                state = self.state[p]
                
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)

                m = state['exp_avg']
                update = (beta1 * m + (1 - beta1) * g).sign()
                p.add_(update, alpha=-lr)
                m.mul_(beta2).add_(g, alpha=1 - beta2)

        return loss


print("Optimizers loaded: RLO, RLO_LambdaA, SmoothLiftedRLO, Lion")

In [ ]:
# =============================================================================
# DATA LOADING
# =============================================================================

class HFDatasetWrapper(Dataset):
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform
        
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image']
        if image.mode != 'RGB':
            image = image.convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, item['label']


def get_imagenet_loaders(data_path, batch_size=256, num_workers=8, image_size=224, use_randaugment=True):
    """Get ImageNet data loaders."""
    MEAN = (0.485, 0.456, 0.406)
    STD = (0.229, 0.224, 0.225)
    
    train_transforms = [
        T.RandomResizedCrop(image_size, interpolation=T.InterpolationMode.BILINEAR),
        T.RandomHorizontalFlip(),
    ]
    if use_randaugment:
        train_transforms.append(T.RandAugment(num_ops=2, magnitude=9))
    train_transforms.extend([T.ToTensor(), T.Normalize(MEAN, STD)])
    train_transform = T.Compose(train_transforms)
    
    val_transform = T.Compose([
        T.Resize(int(image_size * 256 / 224)),
        T.CenterCrop(image_size),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])
    
    try:
        from datasets import load_dataset
        ds = load_dataset("imagenet-1k", cache_dir=str(data_path), trust_remote_code=True)
        train_ds = HFDatasetWrapper(ds['train'], train_transform)
        val_ds = HFDatasetWrapper(ds['validation'], val_transform)
    except:
        train_ds = torchvision.datasets.ImageFolder(Path(data_path)/'train', train_transform)
        val_ds = torchvision.datasets.ImageFolder(Path(data_path)/'val', val_transform)
    
    # Windows: limit workers
    if IS_WINDOWS:
        num_workers = min(num_workers, 4)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True,
                              persistent_workers=num_workers > 0)
    val_loader = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    
    print(f"Train: {len(train_ds)} samples, Val: {len(val_ds)} samples")
    return train_loader, val_loader


class GPUMixup:
    """GPU Mixup/CutMix."""
    def __init__(self, mixup_alpha=1.0, cutmix_alpha=1.0, prob=1.0, switch_prob=0.5, num_classes=1000):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.prob = prob
        self.switch_prob = switch_prob
        self.num_classes = num_classes
        
    @torch.no_grad()
    def __call__(self, x, target):
        if random.random() > self.prob:
            return x, F.one_hot(target, self.num_classes).float()
        
        use_cutmix = random.random() < self.switch_prob
        alpha = self.cutmix_alpha if use_cutmix else self.mixup_alpha
        lam = np.random.beta(alpha, alpha)
        
        B = x.size(0)
        index = torch.randperm(B, device=x.device)
        
        if use_cutmix:
            _, _, H, W = x.shape
            cut_rat = math.sqrt(1.0 - lam)
            cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
            cx, cy = random.randint(0, W), random.randint(0, H)
            x1, y1 = max(cx - cut_w//2, 0), max(cy - cut_h//2, 0)
            x2, y2 = min(cx + cut_w//2, W), min(cy + cut_h//2, H)
            x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
            lam = 1 - ((x2-x1)*(y2-y1) / (W*H))
        else:
            x = lam * x + (1-lam) * x[index]
        
        y = F.one_hot(target, self.num_classes).float()
        y_perm = F.one_hot(target[index], self.num_classes).float()
        return x, lam * y + (1-lam) * y_perm


print("Data loaders ready.")

In [ ]:
# =============================================================================
# MODELS
# =============================================================================

def create_resnet50(num_classes=1000):
    from torchvision.models import resnet50
    return resnet50(weights=None, num_classes=num_classes)


class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = attn_drop
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
    
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        x = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_drop if self.training else 0.0)
        x = x.transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))


class MLP(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads, qkv_bias, attn_drop, drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio), drop=drop)
    
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4., qkv_bias=True,
                 drop_rate=0., attn_drop_rate=0.):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)
        
        self.blocks = nn.ModuleList([Block(embed_dim, num_heads, mlp_ratio, qkv_bias, drop_rate, attn_drop_rate) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = torch.cat((self.cls_token.expand(B, -1, -1), x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x)[:, 0])


def create_vit_small(num_classes=1000):
    return VisionTransformer(embed_dim=384, depth=12, num_heads=6, num_classes=num_classes)

def create_vit_base(num_classes=1000):
    return VisionTransformer(embed_dim=768, depth=12, num_heads=12, num_classes=num_classes)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"ResNet-50: {count_params(create_resnet50())/1e6:.1f}M params")
print(f"ViT-S/16: {count_params(create_vit_small())/1e6:.1f}M params")

In [ ]:
# =============================================================================
# TRAINING ENGINE
# =============================================================================

class CosineScheduler:
    def __init__(self, optimizer, base_lr, total_steps, warmup_steps, min_lr=0):
        self.optimizer = optimizer
        self.base_lr = base_lr
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.min_lr = min_lr
        self.step_count = 0
        
    def step(self):
        self.step_count += 1
        lr = self._get_lr()
        for g in self.optimizer.param_groups:
            g['lr'] = lr
        return lr
    
    def _get_lr(self):
        if self.step_count < self.warmup_steps:
            return self.base_lr * self.step_count / max(1, self.warmup_steps)
        progress = (self.step_count - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
        return self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))


def create_optimizer(model, opt_name, lr, wd):
    """Create optimizer with proper scaling."""
    # Sign-based optimizers need smaller lr, larger wd
    if opt_name in ['lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        lr, wd = lr * 0.1, wd * 10
    
    params = model.parameters()
    if opt_name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    elif opt_name == "lion":
        return Lion(params, lr=lr, weight_decay=wd)
    elif opt_name == "rlo":
        return RLO(params, lr=lr, weight_decay=wd)
    elif opt_name == "rlo_lambda_a":
        return RLO_LambdaA(params, lr=lr, weight_decay=wd)
    elif opt_name == "smooth_lifted_rlo":
        return SmoothLiftedRLO(params, lr=lr, weight_decay=wd)
    raise ValueError(f"Unknown optimizer: {opt_name}")


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = model(images)
            if criterion:
                total_loss += criterion(logits, labels).item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / max(total, 1), 100.0 * correct / max(total, 1)


print("Training engine ready.")

In [ ]:
# =============================================================================
# QUICK TEST: CIFAR-10
# =============================================================================

def quick_test_cifar10(epochs=5, batch_size=512):
    """
    Quick test on CIFAR-10 to verify GPU utilization.
    NO torch.compile - works on Windows!
    """
    print("\n" + "="*70)
    print("QUICK TEST: CIFAR-10 (Windows Compatible)")
    print(f"Batch size: {batch_size}")
    print("="*70)
    
    # Data
    transform_train = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])
    transform_test = T.Compose([
        T.ToTensor(),
        T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])
    
    trainset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=transform_test)
    
    # Windows: fewer workers
    num_workers = 0 if IS_WINDOWS else 8
    
    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    test_loader = DataLoader(testset, batch_size=batch_size*2, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    
    print(f"Workers: {num_workers}, Batches: {len(train_loader)}")
    
    # Model
    from torchvision.models import resnet18
    
    def make_model():
        model = resnet18(num_classes=10)
        model.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        model.maxpool = nn.Identity()
        return model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizers = ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']
    results = {}
    
    for opt_name in optimizers:
        print(f"\n--- {opt_name.upper()} ---")
        set_seed(42)
        model = make_model()
        
        # NO torch.compile on Windows!
        
        optimizer = create_optimizer(model, opt_name, lr=1e-3, wd=0.05)
        history = {'train_acc': [], 'test_acc': [], 'throughput': []}
        
        for epoch in range(epochs):
            model.train()
            epoch_start = time.time()
            correct, total = 0, 0
            
            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
            for xb, yb in pbar:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                
                optimizer.zero_grad(set_to_none=True)
                
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    logits = model(xb)
                    loss = criterion(logits, yb)
                
                loss.backward()
                optimizer.step()
                
                pred = logits.argmax(1)
                correct += (pred == yb).sum().item()
                total += yb.size(0)
            
            epoch_time = time.time() - epoch_start
            throughput = len(trainset) / epoch_time
            train_acc = 100 * correct / total
            
            # Eval
            _, test_acc = evaluate(model, test_loader, criterion)
            
            history['train_acc'].append(train_acc)
            history['test_acc'].append(test_acc)
            history['throughput'].append(throughput)
            
            print(f"  Epoch {epoch+1}: Train={train_acc:.1f}%, Test={test_acc:.1f}%, "
                  f"Throughput={throughput:.0f} img/s")
        
        results[opt_name] = {
            'best_acc': max(history['test_acc']),
            'avg_throughput': np.mean(history['throughput']),
            'history': history
        }
    
    # Summary
    print("\n" + "="*70)
    print("RESULTS")
    print("="*70)
    print(f"{'Optimizer':<20} {'Best Acc':>10} {'Throughput':>15}")
    print("-"*45)
    for name, res in results.items():
        print(f"{name:<20} {res['best_acc']:>9.2f}% {res['avg_throughput']:>12.0f} img/s")
    
    return results


# Run test
print("Running quick test (no torch.compile)...")
cifar_results = quick_test_cifar10(epochs=10, batch_size=512)

In [ ]:
# =============================================================================
# EXPERIMENT 4.1: ImageNet
# =============================================================================

def train_imagenet(model_name, opt_name, epochs, batch_size, lr, wd, 
                   warmup_epochs=5, use_randaugment=False, use_mixup=False):
    """Train on ImageNet."""
    set_seed(42)
    run_name = f"{model_name}_{opt_name}_{datetime.now().strftime('%H%M%S')}"
    
    print(f"\n{'='*60}")
    print(f"Training: {run_name}")
    print(f"{'='*60}")
    
    # Data
    train_loader, val_loader = get_imagenet_loaders(
        IMAGENET_PATH, batch_size=batch_size, use_randaugment=use_randaugment
    )
    
    # Model
    if model_name == "resnet50":
        model = create_resnet50()
    elif model_name == "vit_s16":
        model = create_vit_small()
    elif model_name == "vit_b16":
        model = create_vit_base()
    else:
        raise ValueError(f"Unknown model: {model_name}")
    model = model.to(device)
    
    print(f"Model: {model_name}, Params: {count_params(model)/1e6:.1f}M")
    
    # Optimizer
    optimizer = create_optimizer(model, opt_name, lr, wd)
    actual_lr = optimizer.param_groups[0]['lr']
    print(f"Optimizer: {opt_name}, lr={actual_lr:.2e}")
    
    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = len(train_loader) * warmup_epochs
    scheduler = CosineScheduler(optimizer, actual_lr, total_steps, warmup_steps)
    
    # Loss
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    mixup = GPUMixup() if use_mixup else None
    
    history = {'train_acc': [], 'val_acc': [], 'throughput': []}
    best_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        epoch_start = time.time()
        correct, total = 0, 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            if mixup:
                images, labels_mixed = mixup(images, labels)
                use_soft = True
            else:
                labels_mixed = None
                use_soft = False
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits = model(images)
                if use_soft:
                    loss = -torch.sum(F.log_softmax(logits, 1) * labels_mixed, 1).mean()
                else:
                    loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            pred = logits.argmax(1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler._get_lr():.2e}'})
        
        epoch_time = time.time() - epoch_start
        throughput = len(train_loader) * batch_size / epoch_time
        train_acc = 100 * correct / total
        
        _, val_acc = evaluate(model, val_loader, criterion)
        
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['throughput'].append(throughput)
        
        is_best = val_acc > best_acc
        best_acc = max(val_acc, best_acc)
        
        print(f"Epoch {epoch+1}: Train={train_acc:.1f}%, Val={val_acc:.1f}% {'*' if is_best else ''}, "
              f"Throughput={throughput:.0f} img/s")
    
    result = {
        'run_name': run_name,
        'best_acc': best_acc,
        'avg_throughput': np.mean(history['throughput']),
        'history': history
    }
    
    with open(RESULTS_DIR / f"{run_name}.json", 'w') as f:
        json.dump(result, f, indent=2)
    
    return result


def run_experiment_4_1():
    """Run all ImageNet experiments."""
    optimizers = ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']
    results = {}
    
    # ResNet-50
    print("\n" + "="*70)
    print("EXPERIMENT 4.1.1: ResNet-50")
    print("="*70)
    
    for opt in optimizers:
        try:
            results[f'resnet50_{opt}'] = train_imagenet(
                'resnet50', opt, epochs=90, batch_size=512,
                lr=0.1, wd=1e-4, warmup_epochs=5
            )
        except Exception as e:
            print(f"Failed {opt}: {e}")
    
    # ViT-S/16
    print("\n" + "="*70)
    print("EXPERIMENT 4.1.2: ViT-S/16")
    print("="*70)
    
    for opt in optimizers:
        try:
            results[f'vit_s16_{opt}'] = train_imagenet(
                'vit_s16', opt, epochs=300, batch_size=384,
                lr=1e-3, wd=0.05, warmup_epochs=30,
                use_randaugment=True, use_mixup=True
            )
        except Exception as e:
            print(f"Failed {opt}: {e}")
    
    with open(RESULTS_DIR / "experiment_4_1.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    return results

# Uncomment to run:
# results_4_1 = run_experiment_4_1()

In [ ]:
# =============================================================================
# EXPERIMENT 4.3: Diffusion
# =============================================================================

class SimpleUNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=128):
        super().__init__()
        def block(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.SiLU(),
                nn.Conv2d(oc, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.SiLU(),
            )
        self.enc1 = block(in_ch, base_ch)
        self.enc2 = block(base_ch, base_ch*2)
        self.enc3 = block(base_ch*2, base_ch*4)
        self.mid = block(base_ch*4, base_ch*4)
        self.dec3 = block(base_ch*8, base_ch*2)
        self.dec2 = block(base_ch*4, base_ch)
        self.dec1 = block(base_ch*2, base_ch)
        self.final = nn.Conv2d(base_ch, in_ch, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
    
    def forward(self, x, t):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m = self.mid(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up(m), e3], 1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], 1))
        return self.final(d1)


def train_diffusion(opt_name, epochs=30, batch_size=256):
    print(f"\nTraining diffusion: {opt_name}")
    
    transform = T.Compose([T.Resize(64), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
    dataset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
    
    model = SimpleUNet().to(device)
    
    T_steps = 1000
    betas = torch.linspace(0.0001, 0.02, T_steps, device=device)
    alpha_bar = torch.cumprod(1.0 - betas, 0)
    sqrt_ab = torch.sqrt(alpha_bar)
    sqrt_1mab = torch.sqrt(1.0 - alpha_bar)
    
    optimizer = create_optimizer(model, opt_name, lr=3e-4, wd=0.01)
    history = []
    
    for epoch in range(epochs):
        model.train()
        losses = []
        for imgs, _ in tqdm(loader, desc=f"Epoch {epoch+1}", leave=False):
            imgs = imgs.to(device, non_blocking=True)
            t = torch.randint(0, T_steps, (imgs.size(0),), device=device)
            noise = torch.randn_like(imgs)
            noisy = sqrt_ab[t].view(-1,1,1,1) * imgs + sqrt_1mab[t].view(-1,1,1,1) * noise
            
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                pred = model(noisy, t.float() / T_steps)
                loss = F.mse_loss(pred, noise)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        
        avg = np.mean(losses)
        history.append(avg)
        print(f"  Epoch {epoch+1}: Loss={avg:.4f}")
    
    return {'optimizer': opt_name, 'final_loss': history[-1], 'history': history}


def run_experiment_4_3():
    print("\n" + "="*70)
    print("EXPERIMENT 4.3: Diffusion")
    print("="*70)
    results = {}
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        try:
            results[opt] = train_diffusion(opt)
        except Exception as e:
            print(f"Failed {opt}: {e}")
    with open(RESULTS_DIR / "experiment_4_3.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    return results

# Uncomment:
# results_4_3 = run_experiment_4_3()

In [ ]:
# =============================================================================
# EXPERIMENT 4.4: Language Modeling
# =============================================================================

class TransformerLM(nn.Module):
    def __init__(self, vocab_size=10000, d_model=512, n_heads=8, n_layers=6, max_len=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, dropout=0.1, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.embed.weight
        nn.init.trunc_normal_(self.pos, std=0.02)
    
    def forward(self, x):
        B, T = x.shape
        x = self.embed(x) * math.sqrt(self.embed.embedding_dim) + self.pos[:, :T]
        mask = torch.triu(torch.ones(T, T, device=x.device), 1).bool()
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.head(self.ln(x))


def train_lm(opt_name, epochs=10, batch_size=64, seq_len=256):
    print(f"\nTraining LM: {opt_name}")
    vocab_size = 10000
    data = torch.randint(0, vocab_size, (100000,))
    
    def get_batch(bs, sl):
        ix = torch.randint(len(data) - sl, (bs,))
        x = torch.stack([data[i:i+sl] for i in ix]).to(device)
        y = torch.stack([data[i+1:i+sl+1] for i in ix]).to(device)
        return x, y
    
    model = TransformerLM(vocab_size).to(device)
    optimizer = create_optimizer(model, opt_name, lr=1e-4, wd=0.1)
    criterion = nn.CrossEntropyLoss()
    
    history = []
    for epoch in range(epochs):
        model.train()
        losses = []
        for _ in tqdm(range(200), desc=f"Epoch {epoch+1}", leave=False):
            x, y = get_batch(batch_size, seq_len)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits = model(x)
                loss = criterion(logits.view(-1, vocab_size), y.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())
        ppl = math.exp(np.mean(losses))
        history.append(ppl)
        print(f"  Epoch {epoch+1}: PPL={ppl:.2f}")
    
    return {'optimizer': opt_name, 'final_ppl': history[-1], 'history': history}


def run_experiment_4_4():
    print("\n" + "="*70)
    print("EXPERIMENT 4.4: Language Modeling")
    print("="*70)
    results = {}
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        try:
            results[opt] = train_lm(opt)
        except Exception as e:
            print(f"Failed {opt}: {e}")
    with open(RESULTS_DIR / "experiment_4_4.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    return results

# Uncomment:
# results_4_4 = run_experiment_4_4()

In [ ]:
# =============================================================================
# VISUALIZATION
# =============================================================================

def plot_results(results, title):
    colors = {'adamw': 'blue', 'lion': 'orange', 'rlo': 'green',
              'rlo_lambda_a': 'red', 'smooth_lifted_rlo': 'purple'}
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for name, res in results.items():
        h = res.get('history', {})
        c = colors.get(name, 'gray')
        if 'train_acc' in h:
            axes[0].plot(h['train_acc'], label=name, color=c)
        if 'test_acc' in h:
            axes[1].plot(h['test_acc'], label=name, color=c)
    
    axes[0].set_title('Training Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].set_title('Test Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{title.replace(' ', '_').lower()}.png", dpi=150)
    plt.show()


if 'cifar_results' in dir():
    plot_results(cifar_results, "CIFAR-10 Results")

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*70)
print("NOTEBOOK READY (Windows Compatible)")
print("="*70)
print("""
Optimizations enabled:
✓ TF32 (10-30% speedup on RTX 30xx/40xx/50xx)
✓ bfloat16 mixed precision
✓ cudnn.benchmark = True
✓ Large batch sizes (512)
✓ Efficient zero_grad(set_to_none=True)
✗ torch.compile DISABLED (Triton not supported on Windows)

Expected throughput on RTX 5090 (without torch.compile):
- CIFAR-10: ~8,000-12,000 img/s
- ImageNet ResNet-50: ~1,500-2,000 img/s
- ImageNet ViT-S/16: ~800-1,200 img/s

If you want torch.compile, use WSL2 + Linux or native Linux.

To run experiments:
1. Quick test ran above (CIFAR-10)
2. run_experiment_4_1() - ImageNet classification
3. run_experiment_4_3() - Diffusion models
4. run_experiment_4_4() - Language modeling
""")